<a href="https://colab.research.google.com/github/Fish210/3470-Competition-Team-Optimization-Model/blob/main/alliance_optimization_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
##### Imports #####
from google.colab import files
uploaded = files.upload()  # choose your .xlsx file
!pip -q install openpyxl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages

##### Load sheets & Compute points #####
FILE = "/content/FTC_ALLIANCE_ML_MODEL_TRAINING_LT2026(team_match_actions).csv"  # change to input file path
SHEET = "team_match_actions"

df = pd.read_csv(FILE) # Removed sheet_name=SHEET as it's not applicable to CSV files
df = df[df["team"].notna()].copy()

count_cols = [
    "auto_leave",
    "auto_artifact_cl_count","auto_artifact_overflow_count","auto_motif_match_count",
    "tele_artifact_cl_count","tele_artifact_overflow_count","tele_motif_match_count","tele_depot_count",
]

for c in count_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

# Scoring constants (locked values for decode season)
df["auto_points"] = (
    3*df["auto_leave"]
    + 3*df["auto_artifact_cl_count"]
    + 1*df["auto_artifact_overflow_count"]
    + 2*df["auto_motif_match_count"]
)

df["tele_points"] = (
    3*df["tele_artifact_cl_count"]
    + 1*df["tele_artifact_overflow_count"]
    + 2*df["tele_motif_match_count"]
    + 1*df["tele_depot_count"]
)

df["total_points"] = df["auto_points"] + df["tele_points"]

# Dead match (any columns that have no value)
df["dead_match"] = (df["total_points"] == 0).astype(int)

TEAM_US = 3470

##### Compute team stats + FitScore vs Our Team ######
team_stats = df.groupby("team").agg(
    mean_total=("total_points","mean"),
    std_total=("total_points","std"),
    dead_rate=("dead_match","mean"),
    mean_auto=("auto_points","mean"),
    mean_tele=("tele_points","mean"),
    mean_auto_CL=("auto_artifact_cl_count","mean"),
    mean_tele_CL=("tele_artifact_cl_count","mean"),
    mean_auto_motif=("auto_motif_match_count","mean"),
    mean_tele_motif=("tele_motif_match_count","mean"),
).reset_index()

team_stats["std_total"] = team_stats["std_total"].fillna(0)

# "Fingerprint" for redundancy penalty (scoring mix)
def fingerprint(row):
    auto = row["mean_auto"]
    tele = row["mean_tele"]
    motif = row["mean_auto_motif"] + row["mean_tele_motif"]
    cl = row["mean_auto_CL"] + row["mean_tele_CL"]
    vec = np.array([auto, tele, motif, cl], dtype=float)
    if vec.sum() == 0:
        return vec
    return vec / (np.linalg.norm(vec) + 1e-9)

fps = {int(r["team"]): fingerprint(r) for _, r in team_stats.iterrows()}
us_vec = fps.get(TEAM_US, None)

# Basic expected alliance points = our mean + their mean
us_mean = float(team_stats.loc[team_stats["team"]==TEAM_US, "mean_total"].iloc[0])

team_stats["expected_alliance_points"] = us_mean + team_stats["mean_total"]

# Risk penalty (your locked style)
team_stats["risk_penalty"] = 0.6*team_stats["std_total"] + 15*team_stats["dead_rate"]

# Redundancy penalty = similarity to 3470 (higher similarity = more overlap)
def cosine_sim(a,b):
    denom = (np.linalg.norm(a)+1e-9)*(np.linalg.norm(b)+1e-9)
    return float(np.dot(a,b)/denom)

if us_vec is not None:
    team_stats["redundancy"] = team_stats["team"].apply(lambda t: cosine_sim(fps[int(t)], us_vec))
else:
    team_stats["redundancy"] = 0.0

# Coverage bonus: if they boost our weaker phase (auto vs tele)
us_auto = float(team_stats.loc[team_stats["team"]==TEAM_US, "mean_auto"].iloc[0])
us_tele = float(team_stats.loc[team_stats["team"]==TEAM_US, "mean_tele"].iloc[0])
weaker = "auto" if us_auto < us_tele else "tele"

if weaker == "auto":
    team_stats["coverage_bonus"] = (team_stats["mean_auto"] - us_auto).clip(lower=0) * 0.15
else:
    team_stats["coverage_bonus"] = (team_stats["mean_tele"] - us_tele).clip(lower=0) * 0.15

# Fit raw score (tunable weights, safe defaults)
team_stats["fit_raw"] = (
    team_stats["expected_alliance_points"]
    - team_stats["risk_penalty"]
    - 5.0*team_stats["redundancy"]
    + team_stats["coverage_bonus"]
)

# Normalize to 0–100
mn, mx = team_stats["fit_raw"].min(), team_stats["fit_raw"].max()
team_stats["fit_score"] = 100*(team_stats["fit_raw"] - mn) / (mx - mn + 1e-9)

# Rank (exclude us from partner ranks)
team_stats["fit_rank_vs_us"] = team_stats[team_stats["team"]!=TEAM_US]["fit_score"]\
    .rank(ascending=False, method="min")

##### Generate the PDF dashboards (one page per team + global pages) #####

pdf_path = "FTC_Dashboards.pdf"

def why_fit(row):
    reasons = []
    if row["mean_auto"] > us_auto:
        reasons.append(f"Stronger Auto than 3470 (+{row['mean_auto']-us_auto:.1f} auto pts)")
    if row["mean_tele"] > us_tele:
        reasons.append(f"Stronger Tele than 3470 (+{row['mean_tele']-us_tele:.1f} tele pts)")
    if row["dead_rate"] > 0:
        reasons.append(f"Reliability risk: dead_rate={row['dead_rate']:.0%}")
    if row["std_total"] > 8:
        reasons.append(f"High variance: std={row['std_total']:.1f}")
    if row["redundancy"] > 0.85:
        reasons.append("Overlap risk: scoring style very similar to 3470")
    if not reasons:
        reasons.append("Balanced fit: solid points + manageable risk")
    return reasons[:4]

with PdfPages(pdf_path) as pdf:
    # --- Global summary pages ---
    # Heatmap table
    hm = team_stats.copy()
    hm = hm.sort_values("fit_score", ascending=False)
    hm_cols = ["fit_score","expected_alliance_points","mean_total","std_total","dead_rate","mean_auto","mean_tele"]
    hm_show = hm[["team"] + hm_cols].set_index("team")

    plt.figure()
    sns.heatmap(hm_show, annot=False)
    plt.title("Teams vs 3470 — Fit + Key Metrics")
    pdf.savefig(); plt.close()

    # Top partners bar
    top = hm[hm["team"] != TEAM_US].head(10)
    plt.figure()
    sns.barplot(data=top, x="team", y="fit_score")
    plt.title("Top Partners for 3470 — Fit Score")
    plt.xticks(rotation=45)
    pdf.savefig(); plt.close()

    # Mean vs STD scatter
    plt.figure()
    sns.scatterplot(data=hm, x="mean_total", y="std_total", size=(1-hm["dead_rate"]), legend=False)
    plt.title("Ceiling vs Risk (Mean vs STD)")
    pdf.savefig(); plt.close()

    # --- Per-team pages ---
    teams = sorted(df["team"].unique())
    for t in teams:
        row = team_stats[team_stats["team"] == t].iloc[0]

        # Match-by-match points for line chart
        tdf = df[df["team"] == t].sort_values("match_id")
        usdf = df[df["team"] == TEAM_US].sort_values("match_id")

        plt.figure(figsize=(11, 6))

        # Line chart
        ax1 = plt.subplot2grid((2,3),(0,0), colspan=2)
        ax1.plot(tdf["match_id"], tdf["total_points"], marker="o", label=f"Team {t}")
        ax1.plot(usdf["match_id"], usdf["total_points"], marker="o", label=f"Team {TEAM_US}")
        ax1.set_title(f"Team {t} vs {TEAM_US} — Points Over Matches")
        ax1.set_xlabel("Match")
        ax1.set_ylabel("Total Points")
        ax1.legend()

        # Stat box
        ax2 = plt.subplot2grid((2,3),(0,2))
        ax2.axis("off")
        stat_lines = [
            f"Fit Score: {row['fit_score']:.1f}",
            f"Fit Rank vs us: {int(row['fit_rank_vs_us']) if t!=TEAM_US else '-'}",
            f"Exp Alliance Pts: {row['expected_alliance_points']:.1f}",
            f"Mean: {row['mean_total']:.1f}",
            f"STD: {row['std_total']:.1f}",
            f"Dead Rate: {row['dead_rate']:.0%}",
        ]
        ax2.text(0, 1, "\n".join(stat_lines), va="top")

        # Bars (auto/tele/motif/cl)
        ax3 = plt.subplot2grid((2,3),(1,0))
        ax4 = plt.subplot2grid((2,3),(1,1))
        ax5 = plt.subplot2grid((2,3),(1,2))

        # Auto vs Tele bar (team vs us)
        comp = pd.DataFrame({
            "metric":["Auto","Tele"],
            f"{t}":[row["mean_auto"], row["mean_tele"]],
            f"{TEAM_US}":[us_auto, us_tele]
        }).set_index("metric")
        comp.plot(kind="bar", ax=ax3, rot=0)
        ax3.set_title("Avg Auto/Tele Points")

        # Motif and CL comparisons
        row_us = team_stats[team_stats["team"] == TEAM_US].iloc[0]
        motif_team = row["mean_auto_motif"] + row["mean_tele_motif"]
        motif_us = row_us["mean_auto_motif"] + row_us["mean_tele_motif"]
        cl_team = row["mean_auto_CL"] + row["mean_tele_CL"]
        cl_us = row_us["mean_auto_CL"] + row_us["mean_tele_CL"]

        ax4.bar(["Motif", "CL"], [motif_team, cl_team])
        ax4.bar(["Motif", "CL"], [motif_us, cl_us], alpha=0.6)
        ax4.set_title("Avg Motif / CL (Team vs 3470)")

        # Why-fit text
        ax5.axis("off")
        reasons = why_fit(row)
        ax5.text(0, 1, "Why they fit:\n- " + "\n- ".join(reasons), va="top")

        plt.tight_layout()
        pdf.savefig()
        plt.close()

print("Saved PDF:", pdf_path)

from google.colab import files
files.download("FTC_Dashboards.pdf")

Saving FTC_ALLIANCE_ML_MODEL_TRAINING_LT2026(team_match_actions).csv to FTC_ALLIANCE_ML_MODEL_TRAINING_LT2026(team_match_actions).csv
Saved PDF: FTC_Dashboards.pdf


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>